# India — inventory draw probe (PPAC + JODI)

Validates implied stock change from PPAC **production**, **imports/exports** (combined trade workbook), and **consumption** vs JODI `STOCKCH` / `CLOSTLV`.

Raw files:
- `data/raw/india/trade/` — `PT_IMPORT_TMT_H.xlsx` + `PT_import.xls`
- `data/raw/india/production/` — `PT_production_product_H.xls`
- `data/processed/india/india_pt_consumption.parquet`
- `data/processed/jodi/jodi_secondary.parquet`


In [1]:
import sys
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

for p in [Path.cwd(), Path.cwd().parent]:
    if (p / 'analytics').is_dir() and (p / 'scripts').is_dir():
        ROOT = p
        break
else:
    raise RuntimeError('Run from country_oil_scraper or notebooks/')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analytics.india_inventory import (
    build_april_implied_inventory,
    build_closing_stocks_summary,
    build_ppac_closing_stocks_summary,
    build_probe_tables,
    cumulative_draw_since,
)

ANCHOR = '2026-02'
WAR_BASELINE = pd.Timestamp('2026-02-01')
tables = build_probe_tables(ROOT, anchor=ANCHOR)
merged = tables['merged']
jodi = tables['jodi_wide']
draws = tables['cumulative_draws']
draws

,product,flow_draw_kt,level_draw_kt,months,latest_period
0,TOTPRODS,417.0,417.0,1,2026-03
1,TOTPRODS_SYN,331.0,331.0,1,2026-03


## 1. Cumulative draw since end-February (JODI headline)

In [2]:
for prod in ['TOTPRODS', 'GASDIES', 'GASOLINE', 'LPG']:
    d = cumulative_draw_since(jodi, ANCHOR, product=prod)
    print(prod, d)


TOTPRODS {'flow_draw_kt': 417.0, 'level_draw_kt': 417.0, 'months': 1, 'latest_period': '2026-03'}
GASDIES {'flow_draw_kt': 235.0, 'level_draw_kt': 235.0, 'months': 1, 'latest_period': '2026-03'}
GASOLINE {'flow_draw_kt': 3.0, 'level_draw_kt': 3.0, 'months': 1, 'latest_period': '2026-03'}
LPG {'flow_draw_kt': 87.0, 'level_draw_kt': 87.0, 'months': 1, 'latest_period': '2026-03'}


## 2. Closing stocks by product since February 2026 (PPAC-timely)

Same layout as Japan dashboard §8, but **monthly changes** use the latest **PPAC** refinery, trade, and demand (Excel + April PDF flash). PPAC does not publish stock **levels** — **Feb 2026** month-end levels are anchored on JODI **`CLOSTLV`** only.

- **`change_kt_since_feb_ppac`** — sum of PPAC-implied monthly `STOCKCH` (Mar–Apr with current files)
- **`latest_est_kt`** — Feb JODI level + that cumulative change
- **`change_kt_jodi_levels`** — reference: JODI published closing levels through its latest month (often Mar)

Months with demand only (no refinery output) are excluded from the balance.

In [3]:
ppac_stocks = build_ppac_closing_stocks_summary(ROOT, jodi, anchor=ANCHOR)
print(ppac_stocks['source_note'])
display(ppac_stocks['summary'].round(1))
print(
    f"Panel sum change (Feb → {ppac_stocks['latest_ppac_period']}): "
    f"{ppac_stocks['grand_change_ppac_kt']:+.0f} kt (PPAC-implied)"
)
print(f"JODI levels published through: {ppac_stocks['latest_jodi_period']}")

Feb levels: JODI CLOSTLV. Monthly changes: PPAC refinery/trade/demand (PDF + Excel; JODI REFGROUT fill when PPAC production missing).


,feb_2026_kt_jodi_anchor,change_kt_since_feb_ppac,implied_2026-03_kt,implied_2026-04_kt,latest_est_kt,latest_month,jodi_latest_kt,change_kt_jodi_levels,total_change_kt
Gasoline,2771.0,-1612.7,-704.7,-908.0,1158.3,2026-04,2768.0,-3.0,-815.6
LPG,866.0,-408.6,-203.6,-205.0,457.4,2026-04,779.0,-87.0,-815.6
Diesel (HSD),5588.0,-354.6,56.4,-411.0,5233.4,2026-04,5353.0,-235.0,-815.6
Jet fuel,786.0,70.6,41.6,29.0,856.6,2026-04,814.0,28.0,-815.6
Fuel oil,609.0,687.9,494.9,193.0,1296.9,2026-04,628.0,19.0,-815.6
Naphtha,375.0,801.8,204.8,597.0,1176.8,2026-04,322.0,-53.0,-815.6
Other products,1138.0,NaN,NaN,NaN,<NA>,2026-04,1072.0,-66.0,-815.6
Kerosene (non-jet),310.0,NaN,NaN,NaN,<NA>,2026-04,290.0,-20.0,-815.6
Total (panel sum),12443.0,-815.6,-110.6,-705.0,10179.4,2026-04,<NA>,<NA>,-815.6


Panel sum change (Feb → 2026-04): -816 kt (PPAC-implied)
JODI levels published through: 2026-03


In [4]:
total = ppac_stocks['totals_estimated_by_date'].copy().sort_values('date')
recent = total[total['date'] >= '2025-10-01']
fig = px.line(
    recent,
    x='date',
    y='estimated_CLOSTLV',
    title='India total product stocks (kt) — Feb JODI anchor + PPAC-implied path',
    markers=True,
)
fig.add_shape(
    type='line', x0=WAR_BASELINE, x1=WAR_BASELINE,
    y0=0, y1=1, yref='paper',
    line=dict(color='gray', width=1, dash='dash'),
)
fig.add_annotation(
    x=WAR_BASELINE, y=1.02, xref='x', yref='paper',
    text='Feb 2026 baseline', showarrow=False,
    font=dict(color='gray', size=11),
)
fig.update_layout(height=420, template='plotly_white', yaxis_title='kt')
fig.show()

In [5]:
flows = ppac_stocks['ppac_flows']
mom = flows[flows['date'] >= '2025-10-01'].copy()
fig2 = px.bar(
    mom,
    x='date',
    y='implied_STOCKCH_ppac',
    color='product_label',
    barmode='relative',
    title='Monthly stock change by product (kt) — PPAC-implied (+ = build)',
)
fig2.update_layout(height=460, template='plotly_white', yaxis_title='kt')
fig2.show()

### 2b. JODI closing levels (reference — lags PPAC)

Published **`CLOSTLV`** only; latest month is typically one behind PPAC trade/demand.

In [6]:
jodi_stocks = build_closing_stocks_summary(jodi, anchor=ANCHOR)
display(jodi_stocks['summary'].round(1))
print(
    f"JODI level change (Feb → {jodi_stocks['latest_period']}): "
    f"{jodi_stocks['grand_change_kt']:+.0f} kt"
)

c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\analytics\india_inventory.py:417: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stk["delta_CLOSTLV"] = stk.groupby("jodi_product")["CLOSTLV"].diff()


,feb_2026_kt,latest_kt,change_kt_since_feb2026,latest_month,total_change_kt
Diesel (HSD),5588.0,5353.0,-235.0,2026-03,-417.0
LPG,866.0,779.0,-87.0,2026-03,-417.0
Other products,1138.0,1072.0,-66.0,2026-03,-417.0
Naphtha,375.0,322.0,-53.0,2026-03,-417.0
Kerosene (non-jet),310.0,290.0,-20.0,2026-03,-417.0
Gasoline,2771.0,2768.0,-3.0,2026-03,-417.0
Fuel oil,609.0,628.0,19.0,2026-03,-417.0
Jet fuel,786.0,814.0,28.0,2026-03,-417.0
Total products,12443.0,12026.0,-417.0,2026-03,-417.0


JODI level change (Feb → 2026-03): -417 kt


## 3. PPAC flows vs JODI (HSD / diesel)

Stable few-% gaps → trade/production parsers align with JODI definitions.

In [7]:
hsd = merged[merged['jodi_product'] == 'GASDIES'].copy()
hsd = hsd.dropna(subset=['ppac_refgrout', 'REFGROUT'])
cols = ['TIME_PERIOD', 'ppac_refgrout', 'REFGROUT', 'ppac_imports', 'TOTIMPSB',
        'ppac_exports', 'TOTEXPSB']
hsd[cols].tail(12)

,TIME_PERIOD,ppac_refgrout,REFGROUT,ppac_imports,TOTIMPSB,ppac_exports,TOTEXPSB
273,2024-10,9426.676198,9489.0,8.651698,4.0,2349.084948,2349.0
274,2024-11,9671.776232,9734.0,4.212751,3.0,2288.864157,2289.0
275,2024-12,10564.701443,10626.0,1.529537,4.0,2489.009997,2489.0
276,2025-01,10770.618803,10815.0,2.646796,4.0,2788.733562,2789.0
277,2025-02,9565.837285,9616.0,7.202736,4.0,2329.134550,2329.0
278,2025-03,10555.582249,10618.0,2.474294,4.0,2825.433269,2504.0
279,2025-04,9463.454952,9526.0,1.146480,4.0,1295.716117,1296.0
280,2025-05,10224.281544,10289.0,2.162070,3.0,2347.982390,2348.0
281,2025-06,9994.436794,10051.0,2.168575,3.0,2245.175744,1807.0
282,2025-07,10492.781020,10468.0,0.664680,3.0,2242.242365,2234.0


In [8]:
plot_df = hsd.dropna(subset=['STOCKCH', 'implied_STOCKCH_jodi_dem'])
fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_df['TIME_PERIOD'].astype(str), y=plot_df['STOCKCH'],
                         name='JODI STOCKCH', mode='lines+markers'))
fig.add_trace(go.Scatter(x=plot_df['TIME_PERIOD'].astype(str), y=plot_df['implied_STOCKCH_jodi_dem'],
                         name='Implied (PPAC supply, JODI demand)', mode='lines+markers'))
fig.update_layout(title='HSD — implied vs JODI stock change (kt)', xaxis_title='Month', yaxis_title='kt')
fig.show()

## 4. Gap diagnostics

- `stockch_gap_jodi_dem` = implied (PPAC refinery+trade, JODI demand) − JODI `STOCKCH`
- `stockch_gap` using PPAC demand shows the bunker/refinery-fuel wedge

In [9]:
from analytics.india_inventory import gap_diagnostics_summary

gap_diagnostics_summary(merged)

stockch_gap_jodi_dem               gap_ppac_dem              
                                mean     std count         mean     std count
jodi_product                                                                 
GASDIES                       -513.9  1994.9   180       -451.9  1984.4   180
GASOLINE                      -439.0   907.6   180       -440.3   907.4   180
KEROSENE_NONJET                 34.7    93.0   180         40.0    89.2   180
LPG                              9.8   272.6   180          7.9   271.3   180

## 5. April 2026 — implied inventory (PPAC demand + PDF trade/production)

JODI is not available for April yet; this uses **PPAC consumption** and flash **PDFs** for supply.

Sign convention (JODI): **positive `implied_STOCKCH` = stock build**; **positive `implied_draw` = stock draw**.

In [10]:
april = build_april_implied_inventory(ROOT, period='2026-04')
display(april['headline'])
april['by_jodi_product']

,period,products_included,refgrout_kt,imports_kt,exports_kt,demand_kt,implied_STOCKCH_kt,implied_draw_kt,note
0,2026-04,8,22346.0,2336.0,3449.0,19295.0,1938.0,-1938.0,Product-only balance; crude oil trade excluded...


,date_x,jodi_product,metric_type_x,refgrout_kt,date_y,metric_type_y,trade_flow_x,imports_kt,date,metric_type,trade_flow_y,exports_kt,demand_kt,TIME_PERIOD,implied_STOCKCH_kt,implied_draw_kt
1,2026-04-01,GASOLINE,REFGROUT,3663.0,2026-04-01,TOTIMPSB,imports,0.0,2026-04-01,TOTEXPSB,exports,887.0,3684.0,2026-04,-908.0,908.0
0,2026-04-01,GASDIES,REFGROUT,9545.0,2026-04-01,TOTIMPSB,imports,3.0,2026-04-01,TOTEXPSB,exports,1627.0,8332.0,2026-04,-411.0,411.0
4,2026-04-01,LPG,REFGROUT,1344.0,2026-04-01,TOTIMPSB,imports,698.0,2026-04-01,TOTEXPSB,exports,35.0,2212.0,2026-04,-205.0,205.0
2,2026-04-01,JETKERO,REFGROUT,1226.0,2026-04-01,TOTIMPSB,imports,0.0,2026-04-01,TOTEXPSB,exports,426.0,771.0,2026-04,29.0,-29.0
3,2026-04-01,KEROSENE_NONJET,REFGROUT,93.0,2026-04-01,TOTIMPSB,imports,0.0,2026-04-01,TOTEXPSB,exports,1.0,28.0,2026-04,64.0,-64.0
7,2026-04-01,RESFUEL,REFGROUT,759.0,2026-04-01,TOTIMPSB,imports,62.0,2026-04-01,TOTEXPSB,exports,138.0,490.0,2026-04,193.0,-193.0
5,2026-04-01,NAPHTHA,REFGROUT,1596.0,2026-04-01,TOTIMPSB,imports,84.0,2026-04-01,TOTEXPSB,exports,324.0,759.0,2026-04,597.0,-597.0
6,2026-04-01,ONONSPEC_BASKET,REFGROUT,4120.0,2026-04-01,TOTIMPSB,imports,1489.0,2026-04-01,TOTEXPSB,exports,11.0,3019.0,2026-04,2579.0,-2579.0


In [11]:
fig = px.bar(
    april['by_jodi_product'],
    x='jodi_product',
    y='implied_STOCKCH_kt',
    title='April 2026 implied stock change by product (kt) — PPAC only',
    labels={'implied_STOCKCH_kt': 'Stock change (kt, + = build)'},
)
fig.add_hline(y=0, line_dash='dot')
fig.show()